# Stream Customers Data From Cloud Files to Delta Lake
1. Read Files from cloud storage using DataStreamReader API
2. Transform the dataframe to add the following columns
  - File Path: Cloud file path
  - Ingestion date: Current Timestamp
3. Write the transformed data stream to Delta Lake Table

## 1. Read files using DataStreamReader API

In [0]:
from pyspark.sql.types import *

customers_schema = StructType(fields=[
    StructField("customer_id", IntegerType()),
    StructField("customer_name", StringType()),
    StructField("date_of_birth", DateType()),
    StructField("telephone", StringType()),
    StructField("email", StringType()),
    StructField("member_since", DateType()),
    StructField("created_timestamp", TimestampType())
])


In [0]:
customers_df = (spark.readStream
                    .format("json")
                    .schema(customers_schema)
                    .load("/Volumes/gizmobox/landing/operational_data/customers_stream/")
)

## 2. Transform the dataframe to add the following columns
- File Path: Cloud file path
- Ingestion date: Current Timestamp

In [0]:
from pyspark.sql.functions import * 

customers_transformed_df = (customers_df.withColumn("file_path", col("_metadata.file_path"))
                                        .withColumn("ingestion_date", current_timestamp()))
                                                   

## 3. Write the transformed data stream to Delta Lake Table

In [0]:
streaming_query = (
    customers_transformed_df.writeStream 
                    .format("delta") 
                    .option("checkpointLocation", "/Volumes/gizmobox/landing/operational_data/customers_stream/_checkpoint_stream")
                    .toTable("gizmobox.bronze.customers_stream")
)

In [0]:
streaming_query.stop()


In [0]:
%sql
SELECT * FROM gizmobox.bronze.customers_stream

customer_id,customer_name,date_of_birth,telephone,email,member_since,created_timestamp,file_path,ingestion_date
6973,Tracy Cole,1998-07-24,+1 5515836612,tony46@mail.com,2024-12-16,2024-12-25T11:06:35Z,dbfs:/Volumes/gizmobox/landing/operational_data/customers_stream/customers_2024_12.json,2025-11-07T23:22:51.68Z
3532,Holly Wilkinson,2003-03-06,null,lindsey61@yahoo.com,2024-12-22,2024-12-26T09:29:16Z,dbfs:/Volumes/gizmobox/landing/operational_data/customers_stream/customers_2024_12.json,2025-11-07T23:22:51.68Z
1211,Jeremy Ball,2002-03-26,+1 5793543882,megan86@yahoo.com,2024-12-01,2024-12-16T16:54:10Z,dbfs:/Volumes/gizmobox/landing/operational_data/customers_stream/customers_2024_12.json,2025-11-07T23:22:51.68Z
7829,Daniel Black,1998-04-13,+1 0933795082,sandra54@example.org,2024-11-16,2024-12-07T18:09:58Z,dbfs:/Volumes/gizmobox/landing/operational_data/customers_stream/customers_2024_12.json,2025-11-07T23:22:51.68Z
6384,Jennifer Carrillo,1994-12-18,+1 5592179843,james34@yahoo.com,2024-12-22,2024-12-24T14:46:04Z,dbfs:/Volumes/gizmobox/landing/operational_data/customers_stream/customers_2024_12.json,2025-11-07T23:22:51.68Z
2639,Jessica Deleon,1995-06-04,+1 1304435999,jeremiah96@gmail.com,2024-11-22,2024-12-21T07:08:28Z,dbfs:/Volumes/gizmobox/landing/operational_data/customers_stream/customers_2024_12.json,2025-11-07T23:22:51.68Z
3084,Tyler Simmons,2003-02-20,null,brian94@mail.com,2024-12-02,2024-12-20T12:32:29Z,dbfs:/Volumes/gizmobox/landing/operational_data/customers_stream/customers_2024_12.json,2025-11-07T23:22:51.68Z
7997,Patricia Abbott,1997-07-31,+1 4053366448,angela97@mail.com,2024-11-28,2024-12-22T23:00:48Z,dbfs:/Volumes/gizmobox/landing/operational_data/customers_stream/customers_2024_12.json,2025-11-07T23:22:51.68Z
9687,Micheal Perez MD,2002-07-18,+1 3079718622,christopher65@example.org,2024-11-22,2024-12-05T08:58:52Z,dbfs:/Volumes/gizmobox/landing/operational_data/customers_stream/customers_2024_12.json,2025-11-07T23:22:51.68Z
2141,Zachary Lopez,2002-09-20,+1 7431190983,glenn87@example.org,2024-12-14,2024-12-17T15:05:15Z,dbfs:/Volumes/gizmobox/landing/operational_data/customers_stream/customers_2024_12.json,2025-11-07T23:22:51.68Z
